In [21]:
import nltk 
from nltk.parse import RecursiveDescentParser, EarleyChartParser
from nltk import CFG, Nonterminal
from nltk.tree import Tree


Grammer Defined 

In [22]:
grammar = CFG.fromstring("""
S -> NP VP
S -> NP VP PP
NP -> Det N
NP -> Det Adj N
VP -> V NP
VP -> V NP PP
VP -> V Adj
PP -> P NP
Det -> "the" | "a"
Adj -> "big" | "small" | "red" | "quick"
N -> "dog" | "cat" | "rug" | "man" | "park"
V -> "chased" | "sat" | "walked" | "saw"
Adv -> "quickly" | "slowly"
P -> "in" | "on" | "by" | "with"
""")

In [23]:
print(grammar)

Grammar with 29 productions (start state = S)
    S -> NP VP
    S -> NP VP PP
    NP -> Det N
    NP -> Det Adj N
    VP -> V NP
    VP -> V NP PP
    VP -> V Adj
    PP -> P NP
    Det -> 'the'
    Det -> 'a'
    Adj -> 'big'
    Adj -> 'small'
    Adj -> 'red'
    Adj -> 'quick'
    N -> 'dog'
    N -> 'cat'
    N -> 'rug'
    N -> 'man'
    N -> 'park'
    V -> 'chased'
    V -> 'sat'
    V -> 'walked'
    V -> 'saw'
    Adv -> 'quickly'
    Adv -> 'slowly'
    P -> 'in'
    P -> 'on'
    P -> 'by'
    P -> 'with'


In [24]:
rd = RecursiveDescentParser(grammar)

In [25]:
sentences = [
    "the dog chased the cat",
    "the quick dog saw a red cat",
    "the big dog sat on the red rug"
]

Bottom Down Parser 

In [26]:
for sent in sentences:
    tokens = sent.lower().split()
    print(f"Sentence : {sent}")
    print("=" * 60)
    found =  True
    for tree in rd.parse(tokens):
        found = True 
        print(tree.pformat())

    if not found :
        print("not a valid parse")

Sentence : the dog chased the cat
(S (NP (Det the) (N dog)) (VP (V chased) (NP (Det the) (N cat))))
Sentence : the quick dog saw a red cat
(S
  (NP (Det the) (Adj quick) (N dog))
  (VP (V saw) (NP (Det a) (Adj red) (N cat))))
Sentence : the big dog sat on the red rug


Top Down Parser Recursive Descent Parser 

In [27]:
for sent in sentences:
    tokens = sent.lower().split()
    print(f"Shift-Reduce Parse : {sent}")
    print("=" * 60)
    found =  False
    for tree in rd.parse(tokens):
        found = True 
        print(tree.pformat())
        print(tree.label())
        for subtree in tree :
            if isinstance(subtree,Tree):
                print(f" {subtree.label()} -> {[leaf for leaf in subtree.leaves()]}")


    if not found :
        print("parse not found")


Shift-Reduce Parse : the dog chased the cat
(S (NP (Det the) (N dog)) (VP (V chased) (NP (Det the) (N cat))))
S
 NP -> ['the', 'dog']
 VP -> ['chased', 'the', 'cat']
Shift-Reduce Parse : the quick dog saw a red cat
(S
  (NP (Det the) (Adj quick) (N dog))
  (VP (V saw) (NP (Det a) (Adj red) (N cat))))
S
 NP -> ['the', 'quick', 'dog']
 VP -> ['saw', 'a', 'red', 'cat']
Shift-Reduce Parse : the big dog sat on the red rug
parse not found


In [28]:
import nltk

# ==========================================
# 1. DEFINE CONTEXT-FREE GRAMMAR (CFG)
# ==========================================
# This grammar defines the syntactic rules for our textual data.
grammar = nltk.CFG.fromstring("""
  S -> NP VP
  NP -> Det N | N
  VP -> V NP | V
  Det -> 'the' | 'a'
  N -> 'dog' | 'cat' | 'mouse'
  V -> 'chased' | 'saw' | 'bitten'
""")

def perform_syntactic_analysis(text):
    print(f"Analyzing Text: '{text}'")
    print("=" * 40)
    
    # Tokenize the input text
    tokens = text.lower().split()
    print(f"Tokens: {tokens}\n")

    # ==========================================
    # 2. TOP-DOWN PARSER (Recursive Descent)
    # ==========================================
    print("--- TOP-DOWN PARSER RESULTS ---")
    rd_parser = nltk.RecursiveDescentParser(grammar)
    
    td_success = False
    for tree in rd_parser.parse(tokens):
        tree.pretty_print()  # Visually prints the syntax tree
        td_success = True
        
    if not td_success:
        print("[INVALID] Top-Down Parser could not form a valid syntax tree.\n")

    print("\n" + "=" * 40 + "\n")

    # ==========================================
    # 3. BOTTOM-UP PARSER (Shift-Reduce)
    # ==========================================
    print("--- BOTTOM-UP PARSER RESULTS ---")
    # trace=2 allows you to see the stack shift/reduce operations in the console
    sr_parser = nltk.ShiftReduceParser(grammar, trace=2) 
    
    bu_success = False
    for tree in sr_parser.parse(tokens):
        print("\nFinal Parse Tree:")
        tree.pretty_print()
        bu_success = True
        
    if not bu_success:
        print("\n[INVALID] Bottom-Up Parser could not reduce to Start Symbol (S).")


# ==========================================
# 4. EXECUTION
# ==========================================
if __name__ == "__main__":
    # Test with a valid sentence based on the CFG above
    test_text = "the dog chased a cat"
    
    perform_syntactic_analysis(test_text)

Analyzing Text: 'the dog chased a cat'
Tokens: ['the', 'dog', 'chased', 'a', 'cat']

--- TOP-DOWN PARSER RESULTS ---
              S               
      ________|_____           
     |              VP        
     |         _____|___       
     NP       |         NP    
  ___|___     |      ___|___   
Det      N    V    Det      N 
 |       |    |     |       |  
the     dog chased  a      cat



--- BOTTOM-UP PARSER RESULTS ---
Parsing 'the dog chased a cat'
    [ * the dog chased a cat]
  S [ 'the' * dog chased a cat]
  R [ Det * dog chased a cat]
  S [ Det 'dog' * chased a cat]
  R [ Det N * chased a cat]
  R [ NP * chased a cat]
  S [ NP 'chased' * a cat]
  R [ NP V * a cat]
  R [ NP VP * a cat]
  R [ S * a cat]
  S [ S 'a' * cat]
  R [ S Det * cat]
  S [ S Det 'cat' * ]
  R [ S Det N * ]
  R [ S NP * ]

[INVALID] Bottom-Up Parser could not reduce to Start Symbol (S).
